# Blood Cell Classification using Deep Learning
## CNN (EfficientNet-B0) vs Vision Transformer (ViT)

**Nolan CACHEUX & Amine EL BAKALI**

### Table of Contents
1. [Introduction & Medical Context](#1-introduction--medical-context)
2. [Dataset Loading & Exploration](#2-dataset-loading--exploration)
3. [Data Preparation](#3-data-preparation)
4. [Architecture 1: CNN (EfficientNet-B0)](#4-architecture-1-cnn-efficientnet-b0)
5. [Architecture 2: Vision Transformer (ViT)](#5-architecture-2-vision-transformer-vit)
6. [Training](#6-training)
7. [Evaluation & Comparison](#7-evaluation--comparison)
8. [Explainability: Grad-CAM](#8-explainability-grad-cam)
9. [Comparison with State of the Art](#9-comparison-with-state-of-the-art)
10. [Conclusion & Discussion](#10-conclusion--discussion)
11. [References](#11-references)

## 1. Introduction & Medical Context

### 1.1 What are Blood Cells?

Blood is a vital connective tissue that circulates through the cardiovascular system, performing essential functions including oxygen transport, immune defense, and waste removal. Blood consists of **plasma** (55%) and **cellular components** (45%), known as **formed elements**.

The cellular components of blood can be classified into three main categories:

#### Red Blood Cells (Erythrocytes)
- Most abundant blood cells (~70% of all cells)
- Biconcave disc shape, no nucleus
- Primary function: oxygen transport via hemoglobin
- Lifespan: ~120 days

#### White Blood Cells (Leukocytes)
- Key players in the immune system
- Subdivided into:
  - **Granulocytes:** Neutrophils, Eosinophils, Basophils
  - **Agranulocytes:** Lymphocytes, Monocytes
- Each type has a distinct morphology and function

#### Platelets (Thrombocytes)
- Smallest blood components
- Essential for blood clotting and wound repair

### 1.2 Why Automated Classification?

Manual blood cell classification by pathologists is time-consuming and subject to inter-observer variability. Deep learning offers a way to automate this process with consistent, high-throughput analysis.

In this project, I compare two fundamentally different architectures:
1. **CNN (EfficientNet-B0):** Processes the image through hierarchical convolutional layers
2. **Vision Transformer (ViT):** Splits the image into patches and processes them with self-attention

## 2. Dataset Loading & Exploration

### 2.1 Environment Setup

I start by installing the required dependencies and setting up the environment for reproducibility.

In [ ]:
# Install all required packages
!pip install -q medmnist keras-hub kagglehub

# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import label_binarize
import warnings
import gc

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if tf.config.list_physical_devices('GPU'):
    print(f"GPU device: {tf.config.list_physical_devices('GPU')[0].name}")
import keras_hub
print(f"Keras Hub version: {keras_hub.__version__}")


### 2.2 Load BloodMNIST Dataset

I'm using the **MedMNIST** library to load the BloodMNIST dataset. This dataset contains 17,092 microscopy images of blood cells from the Barcelona Hospital, originally published by Acevedo et al. (2020).

**Dataset Specifications:**
- **Image Size:** 224x224x3 (RGB)
- **Classes:** 8 blood cell types
- **Split:** Train (11,959), Validation (1,712), Test (3,421)
- **Source:** Peripheral blood smear images

In [ ]:
import medmnist
from medmnist import INFO, Evaluator

# Dataset information
data_flag = 'bloodmnist'
info = INFO[data_flag]

CLASS_NAMES = [
    'Basophil',
    'Eosinophil',
    'Erythroblast',
    'Immature Granulocyte',
    'Lymphocyte',
    'Monocyte',
    'Neutrophil',
    'Platelet'
]

print(f"Dataset: {info['description']}")
print(f"Classes: {len(info['label'])} types")

# Load at 128x128 (good balance between memory and resize speed)
DataClass = getattr(medmnist, info['python_class'])

train_dataset = DataClass(split='train', download=True, size=128)
val_dataset = DataClass(split='val', download=True, size=128)
test_dataset = DataClass(split='test', download=True, size=128)

print(f"Split sizes: Train={len(train_dataset):,} | Val={len(val_dataset):,} | Test={len(test_dataset):,}")

### 2.3 Extract and Normalize

Extracting images into NumPy arrays and normalizing to [0, 1] in one step to save memory. Images are loaded at 128x128 and resized to 224x224 in the data pipeline.

In [ ]:
# Extract images and labels into NumPy arrays
def extract_data(dataset):
    images = []
    labels = []
    for img, label in dataset:
        images.append(np.array(img))
        labels.append(label)
    return np.array(images, dtype=np.float32) / 255.0, np.array(labels).flatten()

# Extract and normalize in one step to save memory
X_train_norm, y_train = extract_data(train_dataset)
X_val_norm, y_val = extract_data(val_dataset)
X_test_norm, y_test = extract_data(test_dataset)

# Free the dataset objects
del train_dataset, val_dataset, test_dataset
import gc; gc.collect()

print(f"X_train: {X_train_norm.shape} | X_val: {X_val_norm.shape} | X_test: {X_test_norm.shape}")
print(f"Pixel range: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}], dtype: {X_train_norm.dtype}")

### 2.4 Class Distribution Analysis

Understanding the class distribution is important to spot potential imbalance issues that could affect model performance.

In [ ]:
# Calculate class distribution
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_val, counts_val = np.unique(y_val, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

# Create distribution dataframe for visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = plt.cm.Set3(np.linspace(0, 1, 8))

# Training set distribution
ax1 = axes[0]
bars1 = ax1.bar(range(8), counts_train, color=colors)
ax1.set_xlabel('Class', fontsize=12)
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.set_title('Training Set Distribution', fontsize=14, fontweight='bold')
ax1.set_xticks(range(8))
ax1.set_xticklabels([f'{i}' for i in range(8)], fontsize=10)

# Validation set distribution
ax2 = axes[1]
bars2 = ax2.bar(range(8), counts_val, color=colors)
ax2.set_xlabel('Class', fontsize=12)
ax2.set_title('Validation Set Distribution', fontsize=14, fontweight='bold')
ax2.set_xticks(range(8))
ax2.set_xticklabels([f'{i}' for i in range(8)], fontsize=10)

# Test set distribution
ax3 = axes[2]
bars3 = ax3.bar(range(8), counts_test, color=colors)
ax3.set_xlabel('Class', fontsize=12)
ax3.set_title('Test Set Distribution', fontsize=14, fontweight='bold')
ax3.set_xticks(range(8))
ax3.set_xticklabels([f'{i}' for i in range(8)], fontsize=10)

# Add class name legend
fig.legend(bars1, CLASS_NAMES, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Class Distribution Across Splits', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Print imbalance ratio
print(f"Class imbalance ratio (max/min): {max(counts_train)/min(counts_train):.2f}")

### 2.5 Sample Images Visualization

Let me visualize some samples from each class to get a feel for what the different blood cell types look like.

In [ ]:
# Display sample images for each class (2 rows x 8 columns)
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for class_idx in range(8):
    class_indices = np.where(y_train == class_idx)[0]
    sample_indices = np.random.choice(class_indices, size=2, replace=False)

    for row, idx in enumerate(sample_indices):
        ax = axes[row, class_idx]
        ax.imshow(X_train_norm[idx])
        if row == 0:
            ax.set_title(f'{CLASS_NAMES[class_idx]}', fontsize=10, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Dataset Citation

**MedMNIST v2** (Yang et al., 2023):

> Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., & Ni, B. (2023). *MedMNIST v2 - A large-scale lightweight benchmark for 2D and 3D biomedical image classification*. Scientific Data, 10(1), 41. https://doi.org/10.1038/s41597-022-01721-8

**Original Blood Cell Dataset** (Acevedo et al., 2020):

> Acevedo, A., Merino, A., Alferez, S., Molina, A., Boldu, L., & Rodellar, J. (2020). *A dataset of microscopic peripheral blood cell images for development of automatic recognition systems*. Data in Brief, 30, 105474.

## 3. Data Preparation

### 3.1 Normalization

Already handled during data extraction above to avoid keeping two copies of the data in memory.

In [ ]:
# Normalization already done during extraction (cell 2.3)
# X_train_norm, X_val_norm, X_test_norm are already in [0, 1] range
print(f"Normalized range: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")

### 3.2 One-Hot Encoding

Converting integer labels to one-hot encoded vectors for multi-class classification with categorical crossentropy loss.

In [ ]:
# One-hot encode labels
NUM_CLASSES = 8

y_train_onehot = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_onehot = keras.utils.to_categorical(y_val, NUM_CLASSES)
y_test_onehot = keras.utils.to_categorical(y_test, NUM_CLASSES)

print(f"One-hot shapes: train={y_train_onehot.shape}, val={y_val_onehot.shape}, test={y_test_onehot.shape}")
print(f"Example: label {y_train[0]} ({CLASS_NAMES[y_train[0]]}) -> {y_train_onehot[0]}")

### 3.3 Data Augmentation

Data augmentation helps prevent overfitting by artificially expanding the training set. I only apply augmentation during training. The pipeline also resizes images from 128x128 to 224x224.

**Pipeline:**
- Resize 128x128 to 224x224
- Random horizontal flip
- Random rotation (+/-10%)
- Random zoom (+/-10%)
- Random contrast adjustment

In [ ]:
# Data augmentation pipeline (includes resize from 128x128 to 224x224)
data_augmentation = keras.Sequential([
    layers.Resizing(224, 224),   # Resize from 128x128 to 224x224
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# Visualize augmented examples
fig, axes = plt.subplots(3, 6, figsize=(15, 8))

sample_idx = np.random.randint(0, len(X_train_norm))
sample_image = X_train_norm[sample_idx:sample_idx+1]
sample_label = y_train[sample_idx]

# Original
axes[0, 0].imshow(sample_image[0])
axes[0, 0].set_title(f'Original\n{CLASS_NAMES[sample_label]}', fontsize=10)
axes[0, 0].axis('off')

# Augmented versions
for i in range(1, 6):
    augmented = data_augmentation(sample_image, training=True)
    axes[0, i].imshow(augmented[0].numpy())
    axes[0, i].set_title(f'Augmented {i}', fontsize=10)
    axes[0, i].axis('off')

# More samples from different classes
for row in range(1, 3):
    sample_idx = np.random.randint(0, len(X_train_norm))
    sample_image = X_train_norm[sample_idx:sample_idx+1]
    sample_label = y_train[sample_idx]

    axes[row, 0].imshow(sample_image[0])
    axes[row, 0].set_title(f'Original\n{CLASS_NAMES[sample_label]}', fontsize=10)
    axes[row, 0].axis('off')

    for i in range(1, 6):
        augmented = data_augmentation(sample_image, training=True)
        axes[row, i].imshow(augmented[0].numpy())
        axes[row, i].set_title(f'Augmented {i}', fontsize=10)
        axes[row, i].axis('off')

plt.suptitle('Data Augmentation Examples (128x128 -> 224x224 + augmentation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Create TensorFlow Datasets

Creating `tf.data.Dataset` objects for efficient data loading and batching.

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
IMG_SIZE = 224
AUTOTUNE = tf.data.AUTOTUNE

# Resize layer for validation/test (no augmentation, just resize)
resize_layer = layers.Resizing(IMG_SIZE, IMG_SIZE)

# Create tf.data.Dataset objects
train_ds = tf.data.Dataset.from_tensor_slices((X_train_norm, y_train_onehot))
val_ds = tf.data.Dataset.from_tensor_slices((X_val_norm, y_val_onehot))
test_ds = tf.data.Dataset.from_tensor_slices((X_test_norm, y_test_onehot))

# Configure datasets for performance
def prepare_dataset(ds, shuffle=False, augment=False):
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        # Augmentation pipeline includes resize 128->224
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        # Just resize 128->224 for val/test
        ds = ds.map(lambda x, y: (resize_layer(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds_augmented = prepare_dataset(train_ds, shuffle=True, augment=True)
val_ds_prepared = prepare_dataset(val_ds)
test_ds_prepared = prepare_dataset(test_ds)

print(f"Batches: train={len(train_ds_augmented)} (augmented) | val={len(val_ds_prepared)} | test={len(test_ds_prepared)} | batch_size={BATCH_SIZE}")

## 4. Architecture 1: CNN (EfficientNet-B0)

### 4.1 Architecture Overview

**EfficientNet-B0** uses a compound scaling method that uniformly scales depth, width, and resolution. It achieves excellent accuracy with fewer parameters than traditional models like ResNet.

**Transfer Learning Strategy:**
1. Load EfficientNet-B0 pre-trained on ImageNet (freeze weights)
2. Add a custom classification head:
   - GlobalAveragePooling2D
   - Dense(256, ReLU) + Dropout(0.3)
   - Dense(128, ReLU) + Dropout(0.2)
   - Dense(8, Softmax)

In [ ]:
def build_efficientnet_model(input_shape=(224, 224, 3), num_classes=8):
    """Build EfficientNet-B0 model with transfer learning."""
    # Load pre-trained EfficientNet-B0 (without top classification layer)
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Freeze the base model weights
    base_model.trainable = False

    # Build the model
    inputs = keras.Input(shape=input_shape)

    # EfficientNet expects inputs in [0, 255] range, so we rescale
    x = layers.Rescaling(255.0)(inputs)

    # Pass through base model
    x = base_model(x, training=False)

    # Classification head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Dense(128, activation='relu', name='dense_128')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    # Create model
    model = keras.Model(inputs, outputs, name='EfficientNet_BloodCell')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Build the model
cnn_model = build_efficientnet_model()

# Display model summary
cnn_model.summary()

# Count parameters
total_params = cnn_model.count_params()
trainable_params = sum([keras.backend.count_params(w) for w in cnn_model.trainable_weights])
non_trainable_params = total_params - trainable_params

print(f"\n Parameter Statistics:")
print(f"   Total parameters:         {total_params:,}")
print(f"   Trainable parameters:     {trainable_params:,}")
print(f"   Non-trainable parameters: {non_trainable_params:,}")

### 4.2 Model Architecture Visualization

The architecture consists of:
1. **Input Layer:** 224x224x3 images
2. **Rescaling Layer:** Scale [0,1] to [0,255] for EfficientNet preprocessing
3. **EfficientNet-B0 Backbone:** Pre-trained feature extractor (frozen)
4. **GlobalAveragePooling2D:** Reduce spatial dimensions
5. **Dense Layers:** 256 then 128 neurons with dropout
6. **Output Layer:** 8-class softmax

```
Input (224x224x3)
      |
Rescaling (x255)
      |
EfficientNet-B0 (frozen, 4M params)
      |
GlobalAveragePooling2D
      |
Dense(256) + ReLU + Dropout(0.3)
      |
Dense(128) + ReLU + Dropout(0.2)
      |
Dense(8) + Softmax
```

## 5. Architecture 2: Vision Transformer (ViT)

### 5.1 Vision Transformer Overview

The **Vision Transformer (ViT)** applies the Transformer architecture, originally designed for NLP, directly to image classification. The key insight is that images can be treated as sequences of patches, similar to how text is treated as sequences of tokens.

**ViT Architecture:**
1. **Patch Extraction:** Split image into 16x16 patches (224/16 = 14, so 14x14 = 196 patches)
2. **Patch Embedding:** Linear projection of flattened patches
3. **Position Embedding:** Learnable position encodings
4. **Transformer Encoder:** Multiple layers of Multi-Head Self-Attention + FFN
5. **Classification Head:** MLP on the [CLS] token

**Our approach:** We use a **pretrained ViT-B/16** model from Keras Hub, originally trained on ImageNet-21k and fine-tuned on ImageNet-1k. This provides a strong initialization that allows the model to converge quickly even on smaller medical datasets. We freeze the base model and only train the classification head, similar to our EfficientNet-B0 approach.

**Why pretrained?** Vision Transformers trained from scratch require extremely large datasets (100M+ images) to learn effective patch representations. With only ~12K training images, a from-scratch ViT would fail to converge. Transfer learning solves this by reusing visual features already learned from ImageNet.


In [ ]:
def build_vit_model(input_shape=(224, 224, 3), num_classes=8):
    """Build Vision Transformer using pretrained ViT-B/16 from Keras Hub."""
    # Load pretrained ViT-B/16 backbone (ImageNet)
    backbone = keras_hub.models.ViTBackbone.from_preset("vit_base_patch16_224_imagenet")
    backbone.trainable = False

    inputs = keras.Input(shape=input_shape)

    # Extract features: output shape (batch, num_patches+1, 768)
    features = backbone(inputs)

    # Use CLS token (first token) for classification
    cls_token = features[:, 0, :]  # (batch, 768)

    # Classification head
    x = layers.Dense(256, activation='gelu', name='dense_256')(cls_token)
    x = layers.Dropout(0.3, name='vit_dropout_1')(x)
    x = layers.Dense(128, activation='gelu', name='dense_128')(x)
    x = layers.Dropout(0.2, name='vit_dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='vit_predictions')(x)

    model = keras.Model(inputs, outputs, name='ViT_B16_BloodCell')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Build ViT model
IMG_SIZE = 224
vit_model = build_vit_model(input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Display model summary
vit_model.summary()

# Count parameters
total_params = vit_model.count_params()
trainable_params = sum([keras.backend.count_params(w) for w in vit_model.trainable_weights])
non_trainable_params = total_params - trainable_params

print(f"\n Parameter Statistics:")
print(f"   Total parameters:         {total_params:,}")
print(f"   Trainable parameters:     {trainable_params:,}")
print(f"   Non-trainable parameters: {non_trainable_params:,}")
print(f"\nViT config: ViT-B/16 pretrained (ImageNet), 768-dim CLS token, classification head only")


### 5.2 ViT Architecture Diagram

```
                    Vision Transformer (ViT-B/16) - Pretrained

   Input Image (224x224x3)
         |
   [Pretrained ViT-B/16 Feature Extractor]
   +==================================================+
   | Patch Extraction (16x16 patches -> 196 patches)  |
   | Linear Projection + Position Embedding            |
   | [CLS] + Patch_1 + Patch_2 + ... + Patch_196      |
   |                                                    |
   | Transformer Encoder (12 layers)                   |
   | +----------------------------------------------+  |
   | | Multi-Head Self-Attention (12 heads)         |  |
   | | Add & LayerNorm                              |  |
   | | Feed-Forward Network (768 -> 3072 -> 768)    |  |
   | | Add & LayerNorm                              |  |
   | +----------------------------------------------+  |
   |                                                    |
   | [CLS] Token -> 768-dim feature vector             |
   +==================================================+
         |  (frozen weights)
   Dense(256, GELU) + Dropout(0.3)
         |
   Dense(128, GELU) + Dropout(0.2)
         |
   Dense(8, Softmax)
```

**Key difference with EfficientNet-B0:** While both use transfer learning, they represent fundamentally different architectures. EfficientNet uses hierarchical convolutional feature extraction (local to global), while ViT processes the image as a flat sequence of patches with global self-attention from the first layer.


## 6. Training

### 6.1 Training Configuration

Both models are trained with the following configuration:
- **Epochs:** 50 (with early stopping)
- **Batch size:** 64
- **Optimizer:** Adam (lr=1e-3)
- **Loss:** Categorical Crossentropy
- **Callbacks:**
  - ModelCheckpoint (save best model based on val_loss)
  - EarlyStopping (patience=10)
  - ReduceLROnPlateau (patience=5, factor=0.5)

In [ ]:
# Training configuration
EPOCHS = 25

def get_callbacks(model_name):
    """Create callbacks for training."""
    return [
        callbacks.ModelCheckpoint(
            filepath=f'best_{model_name}.keras',
            monitor='val_loss',
            save_best_only=True,
            mode='min',
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
    ]

print(f"Training config: {EPOCHS} epochs, batch_size={BATCH_SIZE}, early_stopping=10, lr_reduce=5")

### 6.2 Train EfficientNet-B0 (CNN)

In [ ]:
# Train CNN model
cnn_history = cnn_model.fit(
    train_ds_augmented,
    validation_data=val_ds_prepared,
    epochs=EPOCHS,
    callbacks=get_callbacks('cnn'),
    verbose=1
)

print("EfficientNet-B0 training complete!")

### 6.3 Train Vision Transformer (ViT)

In [ ]:
# Train ViT model
vit_history = vit_model.fit(
    train_ds_augmented,
    validation_data=val_ds_prepared,
    epochs=EPOCHS,
    callbacks=get_callbacks('vit'),
    verbose=1
)

print("Vision Transformer training complete!")

### 6.4 Training Curves Visualization

In [ ]:
def plot_training_history(histories, model_names):
    """Plot training curves for multiple models."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    colors = ['#2196F3', '#FF5722']  # Blue for CNN, Orange for ViT

    for idx, (history, name, color) in enumerate(zip(histories, model_names, colors)):
        # Loss
        axes[0, idx].plot(history.history['loss'], label='Train Loss', color=color, linewidth=2)
        axes[0, idx].plot(history.history['val_loss'], label='Val Loss', color=color,
                          linewidth=2, linestyle='--')
        axes[0, idx].set_title(f'{name} - Loss', fontsize=14, fontweight='bold')
        axes[0, idx].set_xlabel('Epoch')
        axes[0, idx].set_ylabel('Loss')
        axes[0, idx].legend()
        axes[0, idx].grid(True, alpha=0.3)

        # Accuracy
        axes[1, idx].plot(history.history['accuracy'], label='Train Accuracy', color=color, linewidth=2)
        axes[1, idx].plot(history.history['val_accuracy'], label='Val Accuracy', color=color,
                          linewidth=2, linestyle='--')
        axes[1, idx].set_title(f'{name} - Accuracy', fontsize=14, fontweight='bold')
        axes[1, idx].set_xlabel('Epoch')
        axes[1, idx].set_ylabel('Accuracy')
        axes[1, idx].legend()
        axes[1, idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot training curves
plot_training_history(
    [cnn_history, vit_history],
    ['EfficientNet-B0 (CNN)', 'Vision Transformer (ViT)']
)

# Print final metrics
print("\n Training Summary:")
print(f"{'Model':<25} {'Best Val Loss':<15} {'Best Val Acc':<15}")
print(f"{'EfficientNet-B0':<25} {min(cnn_history.history['val_loss']):<15.4f} {max(cnn_history.history['val_accuracy']):<15.4f}")
print(f"{'Vision Transformer':<25} {min(vit_history.history['val_loss']):<15.4f} {max(vit_history.history['val_accuracy']):<15.4f}")


## 7. Evaluation & Comparison

### 7.1 Compare Models on Validation Set

Following best practices, I compare both models on the validation set first. Only the best-performing model will be evaluated on the held-out test set for the final unbiased estimate.

In [ ]:
# Load best models
cnn_model.load_weights('best_cnn.keras')
vit_model.load_weights('best_vit.keras')

# Compare on VALIDATION set
cnn_val_loss, cnn_val_acc = cnn_model.evaluate(val_ds_prepared, verbose=0)
vit_val_loss, vit_val_acc = vit_model.evaluate(val_ds_prepared, verbose=0)

print(f"Validation Results:")
print(f"  EfficientNet-B0:      loss={cnn_val_loss:.4f}, accuracy={cnn_val_acc:.4f}")
print(f"  Vision Transformer:   loss={vit_val_loss:.4f}, accuracy={vit_val_acc:.4f}")

# Determine the best model based on validation accuracy
if cnn_val_acc >= vit_val_acc:
    best_model = cnn_model
    best_model_name = "EfficientNet-B0"
    other_model_name = "Vision Transformer"
else:
    best_model = vit_model
    best_model_name = "Vision Transformer"
    other_model_name = "EfficientNet-B0"

print(f"\nBest model on validation: {best_model_name}")

### 7.2 Validation Predictions and Classification Reports

In [ ]:
# Validation predictions for both models (for comparison)
X_val_resized = tf.image.resize(X_val_norm, [IMG_SIZE, IMG_SIZE]).numpy()

cnn_val_predictions = cnn_model.predict(X_val_resized, verbose=0)
vit_val_predictions = vit_model.predict(X_val_resized, verbose=0)

cnn_val_pred_classes = np.argmax(cnn_val_predictions, axis=1)
vit_val_pred_classes = np.argmax(vit_val_predictions, axis=1)

del X_val_resized
gc.collect()

print("EfficientNet-B0 Validation Report:\n")
print(classification_report(y_val, cnn_val_pred_classes, target_names=CLASS_NAMES, digits=4))

print("\nVision Transformer Validation Report:\n")
print(classification_report(y_val, vit_val_pred_classes, target_names=CLASS_NAMES, digits=4))

### 7.3 Confusion Matrices (Validation Set)

In [ ]:
# Compute confusion matrices
cnn_cm = confusion_matrix(y_val, cnn_val_pred_classes)
vit_cm = confusion_matrix(y_val, vit_val_pred_classes)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CNN Confusion Matrix
sns.heatmap(cnn_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('EfficientNet-B0 Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# ViT Confusion Matrix
sns.heatmap(vit_cm, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title('Vision Transformer Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze common misclassifications
print("\n Analysis of Common Misclassifications:")
for model_name, cm in [('EfficientNet-B0', cnn_cm), ('ViT', vit_cm)]:
    print(f"\n{model_name}:")
    # Find top misclassifications
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)
    top_errors = np.unravel_index(np.argsort(cm_copy.ravel())[-3:], cm_copy.shape)
    for true_idx, pred_idx in zip(top_errors[0][::-1], top_errors[1][::-1]):
        if cm_copy[true_idx, pred_idx] > 0:
            print(f"  {CLASS_NAMES[true_idx]} → {CLASS_NAMES[pred_idx]}: {cm_copy[true_idx, pred_idx]} errors")

### 7.4 ROC Curves and AUC Scores (Validation Set)

In [ ]:
# Binarize labels for ROC curves
y_val_bin = label_binarize(y_val, classes=range(NUM_CLASSES))

# Compute ROC curves and AUC for each class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

def plot_roc_curves(predictions, y_true, ax, title, color_base):
    """Plot ROC curves for multi-class classification."""
    colors = plt.cm.get_cmap(color_base)(np.linspace(0.2, 0.8, NUM_CLASSES))

    auc_scores = []
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_true[:, i], predictions[:, i])
        roc_auc = auc(fpr, tpr)
        auc_scores.append(roc_auc)
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f'{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{title}\nMean AUC = {np.mean(auc_scores):.3f}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

    return auc_scores

# Plot ROC curves for both models
cnn_auc_scores = plot_roc_curves(cnn_val_predictions, y_val_bin, axes[0], 'EfficientNet-B0 ROC Curves', 'Blues')
vit_auc_scores = plot_roc_curves(vit_val_predictions, y_val_bin, axes[1], 'Vision Transformer ROC Curves', 'Oranges')

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Print AUC summary
print("\n AUC Scores per Class:")
print(f"{'Class':<25} {'EfficientNet-B0':<15} {'ViT':<15}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<25} {cnn_auc_scores[i]:<15.4f} {vit_auc_scores[i]:<15.4f}")
print(f"{'Mean AUC':<25} {np.mean(cnn_auc_scores):<15.4f} {np.mean(vit_auc_scores):<15.4f}")

### 7.5 Validation Summary and Best Model Selection

In [ ]:
# Calculate summary metrics
def get_metrics(y_true, y_pred, predictions):
    """Calculate comprehensive metrics."""
    accuracy = np.mean(y_true == y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    # AUC (macro average)
    y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    auc_scores = []
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], predictions[:, i])
        auc_scores.append(auc(fpr, tpr))
    mean_auc = np.mean(auc_scores)

    return accuracy, precision, recall, f1, mean_auc

cnn_metrics = get_metrics(y_val, cnn_val_pred_classes, cnn_val_predictions)
vit_metrics = get_metrics(y_val, vit_val_pred_classes, vit_val_predictions)

# Create comparison table
print("VALIDATION COMPARISON")
print(f"\n{'Metric':<20} {'EfficientNet-B0':<20} {'Vision Transformer':<20}")
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Mean AUC']
for i, metric in enumerate(metrics):
    print(f"{metric:<20} {cnn_metrics[i]:<20.4f} {vit_metrics[i]:<20.4f}")

# Highlight winner
winner = "EfficientNet-B0" if cnn_metrics[0] > vit_metrics[0] else "Vision Transformer"
print(f"\n Best Performing Model: {winner}")
print(f"   Accuracy difference: {abs(cnn_metrics[0] - vit_metrics[0]) * 100:.2f}%")

### 7.6 Final Evaluation on Test Set (Best Model Only)

As per the project requirements, I evaluate only the best-performing model on the held-out test set. This gives the final unbiased performance estimate.

In [ ]:
# Final test evaluation - ONLY the best model
X_test_resized = tf.image.resize(X_test_norm, [IMG_SIZE, IMG_SIZE]).numpy()

best_test_loss, best_test_acc = best_model.evaluate(
    tf.data.Dataset.from_tensor_slices((X_test_resized, keras.utils.to_categorical(y_test, NUM_CLASSES))).batch(BATCH_SIZE),
    verbose=0
)

best_test_predictions = best_model.predict(X_test_resized, verbose=0)
best_test_pred_classes = np.argmax(best_test_predictions, axis=1)

print(f"Final Test Evaluation ({best_model_name}):")
print(f"  Test Accuracy: {best_test_acc:.4f}")
print(f"  Test Loss: {best_test_loss:.4f}")

print(f"\n{best_model_name} Test Classification Report:\n")
print(classification_report(y_test, best_test_pred_classes, target_names=CLASS_NAMES, digits=4))

# Save for Grad-CAM and SOTA comparison
test_predictions = best_test_predictions
test_pred_classes = best_test_pred_classes

del X_test_resized
gc.collect()

## 8. Explainability: Grad-CAM

### 8.1 Understanding Model Decisions

**Grad-CAM (Gradient-weighted Class Activation Mapping)** provides visual explanations by highlighting regions of the input image that are most important for the model's prediction.

**Method:**
1. Compute gradients of the class score with respect to feature maps
2. Global average pooling of gradients to get importance weights
3. Weighted combination of feature maps
4. Apply ReLU and normalize

I apply Grad-CAM to the best model ({best_model_name}) on test set samples.

**Reference:** Selvaraju, R.R. et al. (2017). "Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization." ICCV 2017.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap for a given image and model."""
    # Create gradient model
    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Compute gradients
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradients of the class score with respect to the feature map
    grads = tape.gradient(class_channel, conv_outputs)

    # Average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weighted combination of feature maps
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()


def overlay_gradcam(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image."""
    # Resize heatmap to match image size
    heatmap = np.uint8(255 * heatmap)
    heatmap = tf.image.resize(heatmap[..., np.newaxis], (img.shape[0], img.shape[1]))
    heatmap = np.squeeze(heatmap.numpy())

    # Apply colormap
    jet = plt.cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap.astype(int)]

    # Superimpose heatmap on image
    superimposed = jet_heatmap * alpha + img * (1 - alpha)
    superimposed = np.clip(superimposed, 0, 1)

    return superimposed, jet_heatmap

print(" Grad-CAM functions defined successfully!")

### 8.2 Grad-CAM for EfficientNet-B0

For the CNN model, I use the last convolutional layer of EfficientNet-B0 to generate the Grad-CAM visualizations.

In [ ]:
# Grad-CAM for EfficientNet-B0
def gradcam_efficientnet(img_array, model, pred_index=None):
    """Grad-CAM for EfficientNet model with proper layer handling."""
    # Find the EfficientNet base model layer
    efficientnet_layer = None
    for layer in model.layers:
        if 'efficientnet' in layer.name.lower():
            efficientnet_layer = layer
            break

    if efficientnet_layer is None:
        print("Warning: EfficientNet layer not found, using fallback")
        return np.ones((7, 7)) * 0.5

    # Get the last conv layer name from EfficientNet
    last_conv_name = None
    for layer in efficientnet_layer.layers:
        if isinstance(layer, layers.Conv2D):
            last_conv_name = layer.name

    if last_conv_name is None:
        print("Warning: No Conv2D layer found in EfficientNet")
        return np.ones((7, 7)) * 0.5

    # Build a simpler gradient model using the internal EfficientNet layer
    last_conv_output = efficientnet_layer.get_layer(last_conv_name).output
    efficientnet_grad_model = keras.Model(
        inputs=efficientnet_layer.input,
        outputs=last_conv_output
    )

    # Forward pass
    img_tensor = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)

        # Get rescaled input (model rescales to [0,255])
        rescaled = img_tensor * 255.0

        # Get conv features
        conv_outputs = efficientnet_grad_model(rescaled, training=False)

        # Get predictions
        predictions = model(img_tensor, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_score = predictions[:, int(pred_index)]

    # Compute gradients of class score w.r.t. conv outputs
    grads = tape.gradient(class_score, conv_outputs)

    if grads is None:
        # Fallback: use input gradient saliency
        with tf.GradientTape() as tape2:
            tape2.watch(img_tensor)
            preds = model(img_tensor, training=False)
            score = preds[:, int(pred_index)]
        input_grads = tape2.gradient(score, img_tensor)
        saliency = tf.reduce_mean(tf.abs(input_grads), axis=-1)[0]
        saliency = (saliency - tf.reduce_min(saliency)) / (tf.reduce_max(saliency) - tf.reduce_min(saliency) + 1e-8)
        return saliency.numpy()

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

# Select images from different classes for visualization
np.random.seed(42)
sample_indices = []
for class_idx in range(8):
    class_samples = np.where(y_test == class_idx)[0]
    if len(class_samples) > 0:
        sample_indices.append(np.random.choice(class_samples))

# Generate Grad-CAM visualizations for CNN
fig, axes = plt.subplots(3, 8, figsize=(20, 8))

for idx, sample_idx in enumerate(sample_indices[:8]):
    img = X_test_norm[sample_idx]
    img_resized = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]).numpy()
    img_array = np.expand_dims(img_resized, axis=0)

    # Get prediction
    pred = cnn_model.predict(img_array, verbose=0)
    pred_class = np.argmax(pred)
    true_class = y_test[sample_idx]

    # Generate Grad-CAM
    heatmap = gradcam_efficientnet(img_array, cnn_model, pred_class)
    superimposed, jet_heatmap = overlay_gradcam(img_resized, heatmap)

    # Plot original
    axes[0, idx].imshow(img_resized)
    axes[0, idx].set_title(f'True: {CLASS_NAMES[true_class][:6]}', fontsize=9)
    axes[0, idx].axis('off')

    # Plot heatmap
    axes[1, idx].imshow(jet_heatmap)
    axes[1, idx].set_title(f'Pred: {CLASS_NAMES[pred_class][:6]}', fontsize=9)
    axes[1, idx].axis('off')

    # Plot overlay
    axes[2, idx].imshow(superimposed)
    conf = pred[0, pred_class]
    color = 'green' if pred_class == true_class else 'red'
    axes[2, idx].set_title(f'Conf: {conf:.2f}', fontsize=9, color=color)
    axes[2, idx].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Heatmap', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Overlay', fontsize=12, fontweight='bold')

plt.suptitle('Grad-CAM Visualizations - EfficientNet-B0', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_cnn.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.3 Attention Visualization for Vision Transformer

For ViT, I visualize attention patterns to understand which patches the model focuses on. Since my ViT uses standard Keras MultiHeadAttention without returning attention weights, I use a gradient-based approach similar to Grad-CAM.

In [ ]:
def gradcam_vit(img_array, model, pred_index=None):
    """Gradient-based saliency visualization for ViT.

    Uses input gradient saliency to visualize which regions
    the ViT model focuses on for its predictions.
    """
    img_tensor = tf.cast(tf.convert_to_tensor(img_array), tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        predictions = model(img_tensor, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_score = predictions[:, int(pred_index)]

    # Compute input gradients
    grads = tape.gradient(class_score, img_tensor)

    # Take absolute value and average across color channels
    saliency = tf.reduce_mean(tf.abs(grads), axis=-1)[0]

    # Apply smoothing via average pooling for better visualization
    saliency_4d = tf.reshape(saliency, [1, saliency.shape[0], saliency.shape[1], 1])
    saliency_smooth = tf.nn.avg_pool2d(saliency_4d, ksize=8, strides=1, padding='SAME')
    saliency_smooth = tf.squeeze(saliency_smooth)

    # Normalize to [0, 1]
    heatmap = (saliency_smooth - tf.reduce_min(saliency_smooth)) / (
        tf.reduce_max(saliency_smooth) - tf.reduce_min(saliency_smooth) + 1e-8
    )

    return heatmap.numpy()

# Generate attention visualizations for ViT
fig, axes = plt.subplots(3, 8, figsize=(20, 8))

for idx, sample_idx in enumerate(sample_indices[:8]):
    img = X_test_norm[sample_idx]
    img_resized = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]).numpy()
    img_array = np.expand_dims(img_resized, axis=0)

    # Get prediction
    pred = vit_model.predict(img_array, verbose=0)
    pred_class = np.argmax(pred)
    true_class = y_test[sample_idx]

    # Generate attention visualization
    heatmap = gradcam_vit(img_array, vit_model, pred_class)

    # Resize heatmap if needed
    if heatmap.shape[0] != img_resized.shape[0]:
        heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (img_resized.shape[0], img_resized.shape[1]))
        heatmap_resized = np.squeeze(heatmap_resized.numpy())
    else:
        heatmap_resized = heatmap

    superimposed, jet_heatmap = overlay_gradcam(img_resized, heatmap_resized)

    # Plot original
    axes[0, idx].imshow(img_resized)
    axes[0, idx].set_title(f'True: {CLASS_NAMES[true_class][:6]}', fontsize=9)
    axes[0, idx].axis('off')

    # Plot heatmap
    axes[1, idx].imshow(jet_heatmap)
    axes[1, idx].set_title(f'Pred: {CLASS_NAMES[pred_class][:6]}', fontsize=9)
    axes[1, idx].axis('off')

    # Plot overlay
    axes[2, idx].imshow(superimposed)
    conf = pred[0, pred_class]
    color = 'green' if pred_class == true_class else 'red'
    axes[2, idx].set_title(f'Conf: {conf:.2f}', fontsize=9, color=color)
    axes[2, idx].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Saliency', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Overlay', fontsize=12, fontweight='bold')

plt.suptitle('Input Gradient Saliency - Vision Transformer (ViT-B/16)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_vit.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.4 Interpretation of Grad-CAM Results

**EfficientNet-B0 (CNN) Observations:**
- The CNN tends to focus on **local features** such as:
  - Nuclear morphology (shape, segmentation)
  - Granule patterns within the cytoplasm
  - Cell boundaries and edges
- The attention is often **distributed** across multiple regions of the cell, reflecting the hierarchical nature of CNNs

**Vision Transformer (ViT-B/16) Observations:**
- The ViT shows **global attention patterns** that span across patches
- It tends to focus on:
  - Overall cell shape and size
  - Relative position of nucleus within the cell
  - Broader contextual features across the entire image

**Key Difference:**
The CNN learns hierarchical local features (edges -> textures -> shapes) through convolutional filters, while the ViT captures global relationships between all patches simultaneously through self-attention from the very first layer. This fundamental architectural difference explains why both models can be complementary in medical image analysis.


## 9. Comparison with State of the Art

### 9.1 MedMNIST Leaderboard Results

The MedMNIST benchmark provides standardized evaluation results for various methods. Here I compare my models with the published baselines.

**Official MedMNIST Baselines (BloodMNIST, 224x224):**

| Method | ACC | AUC | Year |
|--------|-----|-----|------|
| ResNet-18 (28) | 0.958 | 0.998 | 2023 |
| ResNet-18 (224) | 0.965 | 0.998 | 2023 |
| ResNet-50 (28) | 0.956 | 0.997 | 2023 |
| ResNet-50 (224) | 0.959 | 0.998 | 2023 |
| auto-sklearn | 0.817 | 0.969 | 2023 |
| AutoKeras | 0.913 | 0.993 | 2023 |
| Google AutoML | 0.960 | 0.998 | 2023 |

*Source: MedMNIST v2 (Yang et al., 2023)*

In [ ]:
# Create comparison table with state of the art
sota_results = {
    'ResNet-18 (28x28)': {'ACC': 0.958, 'AUC': 0.998},
    'ResNet-18 (224x224)': {'ACC': 0.965, 'AUC': 0.998},
    'ResNet-50 (28x28)': {'ACC': 0.956, 'AUC': 0.997},
    'ResNet-50 (224x224)': {'ACC': 0.959, 'AUC': 0.998},
    'auto-sklearn': {'ACC': 0.817, 'AUC': 0.969},
    'AutoKeras': {'ACC': 0.913, 'AUC': 0.993},
    'Google AutoML Vision': {'ACC': 0.960, 'AUC': 0.998},
}

# Add our results
our_results = {
    'EfficientNet-B0 (Ours)': {'ACC': cnn_metrics[0], 'AUC': cnn_metrics[4]},
    'ViT (Ours)': {'ACC': vit_metrics[0], 'AUC': vit_metrics[4]},
}

# Create comparison plot
methods = list(sota_results.keys()) + list(our_results.keys())
acc_values = [sota_results.get(m, our_results.get(m))['ACC'] for m in methods]
auc_values = [sota_results.get(m, our_results.get(m))['AUC'] for m in methods]

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Colors: blue for SOTA, orange/green for ours
colors = ['#2196F3'] * len(sota_results) + ['#FF9800', '#4CAF50']

# Accuracy comparison
bars1 = axes[0].barh(methods, acc_values, color=colors)
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy Comparison with State of the Art', fontsize=14, fontweight='bold')
axes[0].set_xlim([0.8, 1.0])
for bar, val in zip(bars1, acc_values):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)

# AUC comparison
bars2 = axes[1].barh(methods, auc_values, color=colors)
axes[1].set_xlabel('AUC', fontsize=12)
axes[1].set_title('AUC Comparison with State of the Art', fontsize=14, fontweight='bold')
axes[1].set_xlim([0.95, 1.0])
for bar, val in zip(bars2, auc_values):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('sota_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print comparison table
print("COMPARISON WITH STATE OF THE ART")
print(f"\n{'Method':<30} {'Accuracy':<12} {'AUC':<12}")
for method, values in {**sota_results, **our_results}.items():
    marker = "" if "Ours" in method else " "
    print(f"{marker} {method:<28} {values['ACC']:<12.3f} {values['AUC']:<12.3f}")

# Analysis
print("\n Performance Analysis:")
best_acc = max(acc_values)
cnn_rank = sorted(acc_values, reverse=True).index(cnn_metrics[0]) + 1
vit_rank = sorted(acc_values, reverse=True).index(vit_metrics[0]) + 1
print(f"  • Best Accuracy: {best_acc:.3f} (ResNet-18 224x224)")
print(f"  • Our EfficientNet-B0 Rank: {cnn_rank}/{len(methods)}")
print(f"  • Our ViT Rank: {vit_rank}/{len(methods)}")

### 9.2 Discussion

**Comparison Analysis:**

1. **EfficientNet-B0 Performance:**
   - Competitive with ResNet baselines on the MedMNIST leaderboard
   - Transfer learning from ImageNet provides strong initialization for medical images
   - Efficient architecture with fewer parameters than ResNet-50

2. **Vision Transformer (ViT-B/16) Performance:**
   - Pretrained ViT achieves strong performance thanks to ImageNet-21k pretraining
   - Global self-attention captures holistic cell features
   - Transfer learning is essential: ViTs trained from scratch on small datasets fail to converge

3. **Architecture Comparison:**
   - **CNN (EfficientNet):** Excels at capturing local patterns through hierarchical convolutions
   - **ViT:** Captures global patch-to-patch relationships through self-attention
   - Both architectures benefit significantly from transfer learning on medical imaging tasks with limited data
   - The complementary nature of local (CNN) vs global (ViT) features suggests potential for ensemble approaches


## 10. Conclusion & Discussion

### 10.1 Summary of Results

In this project, we developed and compared two fundamentally different deep learning architectures for blood cell classification:

| Model | Architecture | Transfer Learning | Key Feature |
|-------|-------------|-------------------|-------------|
| **EfficientNet-B0** | CNN (Convolutional) | ImageNet | Hierarchical local features |
| **ViT-B/16** | Transformer (Self-Attention) | ImageNet-21k | Global patch relationships |

Both models leverage transfer learning, which proved essential for achieving strong performance on the relatively small BloodMNIST dataset (~12K training images).

### 10.2 Key Takeaways

1. **Transfer learning is critical:** Both models achieved strong results by leveraging pretrained weights. Training a ViT from scratch on 12K images would not converge, confirming that Transformers require much larger datasets than CNNs when trained from scratch.

2. **Different but complementary architectures:** EfficientNet captures local morphological features (cell edges, granule patterns) while ViT captures global relationships between image patches. Both perspectives are valuable for medical image analysis.

3. **Grad-CAM reveals different attention patterns:** The CNN focuses on specific cell structures (nucleus, granules), while the ViT distributes attention more globally across the entire cell image.

4. **Medical imaging applicability:** Automated blood cell classification could significantly reduce pathologist workload and improve diagnostic consistency, especially in resource-limited settings.

### 10.3 Limitations and Future Work

- **Fine-tuning:** Unfreezing some layers of the pretrained models could improve performance further
- **Data augmentation:** More aggressive augmentation (CutMix, MixUp) could help both models generalize better
- **Ensemble methods:** Combining CNN and ViT predictions could leverage their complementary strengths
- **Larger datasets:** Testing on full-resolution blood cell images rather than the downscaled MedMNIST version


## 11. References

### Primary References

1. **MedMNIST v2**
   > Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., & Ni, B. (2023). *MedMNIST v2 - A large-scale lightweight benchmark for 2D and 3D biomedical image classification*. Scientific Data, 10(1), 41.
   > https://doi.org/10.1038/s41597-022-01721-8

2. **Vision Transformer (ViT)**
   > Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., Dehghani, M., Minderer, M., Heigold, G., Gelly, S., Uszkoreit, J., & Houlsby, N. (2020). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR 2021.

3. **EfficientNet**
   > Tan, M. & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML 2019.

4. **Grad-CAM**
   > Selvaraju, R.R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*. ICCV 2017.

5. **Blood Cell Dataset**
   > Acevedo, A., Merino, A., Alferez, S., Molina, A., Boldu, L., & Rodellar, J. (2020). *A dataset of microscopic peripheral blood cell images for development of automatic recognition systems*. Data in Brief, 30, 105474.

### Additional References

6. **Attention is All You Need**
   > Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS 2017.

7. **Transfer Learning for Medical Imaging**
   > Raghu, M. et al. (2019). *Transfusion: Understanding Transfer Learning for Medical Imaging*. NeurIPS 2019.